# 05l: Detect Anomalies in Synthetic Telemetry

**Time:** 35–45 minutes. **Resources:** CPU, approximately 1 GB RAM. Build a normal-behavior model, choose an alert threshold on validation labels, then measure missed events and false positives on untouched test rows.

**Objectives:** distinguish an anomaly score from a probability, avoid threshold tuning on test data, and inspect the operational cost of false alerts.

**Prerequisites:** complete [the Jupyter introduction](05_Running_Jupyter_Lab.ipynb), select a Python 3.10+ kernel, and follow the [support guide](README.md). Packages: NumPy, pandas, scikit-learn and Matplotlib. Use your approved environment; this notebook does not install packages. CPU only, no GPU, credentials, gateway, or external service required. Run cells in order, then restart the kernel and Run All to check reproducibility.

## Install the notebook dependencies if needed

The next cell installs the shared CPU packages into **this notebook's selected Python kernel** using `%pip`. It includes **scikit-learn**, whose import name is `sklearn`, plus NumPy, pandas, matplotlib, Pillow, and IPython.

If the packages are missing or this is a fresh environment, set `INSTALL_PACKAGES = True` and run the cell once. Only do this when installation is permitted and pip is configured to use your approved package source. If the environment is already prepared, leave it `False`. The cell finds `requirements.txt` whether the kernel starts in the repository root or `05_Jupyter`.

After a successful installation, change the flag back to `False`, use **Kernel → Restart Kernel**, and run the notebook from the top. A restart clears old imports and variables. If installation fails or the source is blocked, stop here and ask the administrator to prepare the environment or approved mirror; do not bypass TLS checks.

PyTorch/CUDA, interactive widgets, ClearML, and live gateway extras remain optional. See the [support guide](README.md#prepare-one-approved-python-environment) before enabling them. Package installation can use the network; the lesson's default dataset path remains offline.


In [ ]:
from pathlib import Path
import sys

# Change to True only when this kernel needs the section's packages installed.
INSTALL_PACKAGES = False

SETUP_HERE = Path.cwd().resolve()
SETUP_LAB = SETUP_HERE if (SETUP_HERE / "image_lab.py").is_file() else SETUP_HERE / "05_Jupyter"
REQUIREMENTS = SETUP_LAB / "requirements.txt"
if not REQUIREMENTS.is_file():
    raise FileNotFoundError("Start this kernel in the repository root or 05_Jupyter folder with requirements.txt present.")
print("Selected kernel Python:", sys.executable)

if INSTALL_PACKAGES:
    from IPython import get_ipython

    shell = get_ipython()
    if shell is None:
        raise RuntimeError("Run this setup cell in the notebook's IPython Python kernel.")
    # Equivalent to %pip install -r "{REQUIREMENTS}"; targets the selected kernel.
    shell.run_line_magic("pip", f'install -r "{REQUIREMENTS}"')
    print("Check pip output for errors. After success, set INSTALL_PACKAGES = False, restart the kernel, and run from the top.")
else:
    print("Installation skipped. If imports fail, set INSTALL_PACKAGES = True and rerun this cell first.")


In [ ]:
from pathlib import Path
import sys
# Accept a kernel launched at the repository root or in this lesson directory.
cwd = Path.cwd().resolve()
LAB_DIR = cwd if (cwd / "05_Running_Jupyter_Lab.ipynb").is_file() else cwd / "05_Jupyter"
if not LAB_DIR.is_dir():
    raise RuntimeError("Start the kernel from the repository root or 05_Jupyter directory.")
if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Generate a labeled teaching dataset

All records below are **invented**, using a fixed seed, and contain no customer or infrastructure data. The first 600 minutes are known-normal training examples. Later validation and test intervals contain injected spikes plus normal load variation. Labels come from the generator, not a realistic labeling process. Real incidents are not always outliers; normal deployments can also look unusual.

In [ ]:
rng = np.random.default_rng(71)
n = 1200
t = np.arange(n)
load = 0.5 + 0.22*np.sin(2*np.pi*t/120) + rng.normal(0, 0.06, n)
events = pd.DataFrame({
    "timestamp": pd.date_range("2025-01-01", periods=n, freq="min", tz="UTC"),
    "cpu_pct": np.clip(20 + 55*load + rng.normal(0, 4, n), 0, 100),
    "latency_ms": np.maximum(1, 40 + 90*load + rng.normal(0, 9, n)),
    "error_rate": np.clip(0.005 + 0.01*load + rng.normal(0, 0.003, n), 0, 1),
    "is_anomaly": np.zeros(n, dtype=int),
})
for start, end in [(600, 900), (900, 1200)]:
    injected = rng.choice(np.arange(start, end), size=24, replace=False)
    events.loc[injected, "is_anomaly"] = 1
    events.loc[injected, "latency_ms"] += rng.uniform(45, 130, len(injected))
    events.loc[injected, "error_rate"] += rng.uniform(0.005, 0.04, len(injected))
train = events.iloc[:600].copy()
validation = events.iloc[600:900].copy()
test = events.iloc[900:].copy()
FEATURES = ["cpu_pct", "latency_ms", "error_rate"]
assert train.is_anomaly.sum() == 0
assert train.timestamp.max() < validation.timestamp.min() < test.timestamp.min()
print("Rows:", len(train), len(validation), len(test))
print("Validation injected events:", int(validation.is_anomaly.sum()))
events.head()

## 2. Fit on normal training data only

Isolation Forest isolates unusual points with random tree partitions. We negate `score_samples` so **larger values mean more unusual**. These values are not calibrated incident probabilities. Neither timestamps nor labels are model features. We do not use the estimator's built-in contamination cutoff: validation labels determine our alert threshold separately.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

detector = IsolationForest(n_estimators=100, max_samples=256, random_state=71, n_jobs=1)
detector.fit(train[FEATURES])
validation_score = -detector.score_samples(validation[FEATURES])
# Candidate percentiles are declared before reading test scores or labels.
quantiles = [0.80, 0.85, 0.90, 0.95, 0.98]
rows = []
for q in quantiles:
    threshold = float(np.quantile(validation_score, q))
    alerts = validation_score >= threshold
    precision, recall, f1, _ = precision_recall_fscore_support(
        validation.is_anomaly, alerts, average="binary", zero_division=0)
    rows.append({"quantile": q, "threshold": threshold, "precision": precision, "recall": recall, "f1": f1})
threshold_scores = pd.DataFrame(rows).sort_values(["f1", "threshold"], ascending=[False, False])
threshold = float(threshold_scores.iloc[0].threshold)
print(threshold_scores.round(4).to_string(index=False))
print("Locked threshold:", threshold)

**Checkpoint:** why might maximizing F1 be a poor production objective when missing one incident costs much more than reviewing an alert? Validation labels make this threshold selection supervised, even though the detector itself fit without labels.

## 3. Evaluate once and inspect false positives

Test labels are used only here. Report the confusion matrix with explicit row/column meanings, precision, recall, F1 and the false-positive rate among normal test events. An always-normal baseline has no alerts and misses every anomaly; high accuracy alone would reward it on this imbalanced data.

In [ ]:
test_score = -detector.score_samples(test[FEATURES])
test_alert = test_score >= threshold
truth = test.is_anomaly.to_numpy()
tn, fp, fn, tp = confusion_matrix(truth, test_alert, labels=[0, 1]).ravel()
precision, recall, f1, _ = precision_recall_fscore_support(truth, test_alert, average="binary", zero_division=0)
print(pd.DataFrame([[tn, fp], [fn, tp]], index=["Actual normal", "Actual anomaly"], columns=["No alert", "Alert"]))
print(pd.Series({"precision": precision, "recall": recall, "F1": f1,
                 "false-positive rate": fp / (fp + tn),
                 "always-normal accuracy": float((truth == 0).mean())}).round(3))
false_positive = test_alert & (truth == 0)
missed = (~test_alert) & (truth == 1)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(test.timestamp, test_score, label="Anomaly score", linewidth=1)
ax.axhline(threshold, linestyle="--", color="black", label="Validation-selected threshold")
ax.scatter(test.timestamp[false_positive], test_score[false_positive], marker="x", s=65, color="red", label=f"False positives ({fp})")
ax.scatter(test.timestamp[missed], test_score[missed], marker="o", facecolors="none", edgecolors="orange", s=65, label=f"Missed anomalies ({fn})")
ax.set(title="Synthetic telemetry: untouched test interval", xlabel="UTC time", ylabel="Anomaly score (not probability)")
ax.legend()
fig.tight_layout()
plt.show()
print("False-positive rows (may be empty):")
print(test.loc[false_positive, ["timestamp"] + FEATURES].head().to_string(index=False))

## 4. Stress-test assumptions

**Exercise:** create a separate development copy and compare thresholds chosen to meet validation recall of at least 90%. Report if no candidate meets that target. Do not search thresholds against this test interval; generate a genuinely separate held-out sequence after making choices. Then introduce normal workload drift to explore false alerts.

**Limitations:** independent injected spikes are easier than many real incidents; a known-normal training period and perfect labels are strong assumptions. Event-level scores do not measure incident detection delay or grouped-alert burden. Real use requires privacy review, representative historical data, drift monitoring, temporal backtests and an operator-approved alert policy. This notebook sends no alerts.

**Troubleshooting:** an empty false-positive table is a valid outcome, not a failure. If F1 changes, confirm seed, dependency versions, chronological slices and that you did not refit on validation/test. Missing packages should be resolved through the approved support-guide environment, not by installing into an unrelated kernel.

**Save and cleanup:** save your notebook and conclusions to confirmed persistent storage. No external files or services were created. Clear outputs before committing if desired, shut down the kernel and stop the hosted session in the dashboard. Verify save/restore as described in the introduction; saved outputs do not preserve a live model in memory.

[Section index](README.md) · [Next: Explore Fashion Similarity](05m_Explore_Fashion_Similarity.ipynb)